# 자세 추정 전 원판 크기·밝기 증강 비교

두 실험은 같은 YOLO26n 사전학습 가중치에서 시작합니다. 기존 베이스라인(크기 0.5, HSV 밝기 0.4)과 각각 비교합니다. 먼저 크기만 끄고 결과를 살펴본 뒤, 두 번째 실험에서 밝기만 끕니다. 두 실험의 학습 스위치는 처음에 꺼져 있습니다.

## 1. 설정과 증강 미리보기

학습셋의 같은 프레임에 강도 범위의 양 끝을 적용해 봅니다. 가운데 열이 원본에 해당합니다. 미리보기는 한 효과씩 보여 주며 실제 학습은 모자이크와 다른 기본 증강도 함께 사용합니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

from wlcv.config import RUNS
from wlcv.training.train import train_ablation
from wlcv.evaluation.augmentation import evaluate_plates, preview_augmentation
from wlcv.evaluation.report import plot_training_report

baseline = RUNS / "plate_baseline"
scale_run = RUNS / "plate_scale_off"
brightness_run = RUNS / "plate_hsv_v_off"
settings = yaml.safe_load((baseline / "args.yaml").read_text())
dataset = Path(settings["data"]).parent
print({key: settings[key] for key in ["model", "epochs", "batch", "imgsz", "seed", "scale", "hsv_v", "multi_scale"]})
preview_augmentation(dataset / "images/train/b04_sn_f_01_f29.jpg")
plt.show()

## 2. scale 학습

작은 원판(a03), 큰 원판(b02), 크기가 비교적 일정한 C그룹에서 기본 크기 증강이 어떤 역할을 했는지 봅니다. 학습을 시작할 때만 아래 셀의 값을 `True`로 바꾸고 이 셀을 실행하세요. 반환된 실제 폴더에 학습 기록과 `weights/best.pt`가 저장됩니다. 학습 후에는 스위치를 다시 `False`로 돌려두면 같은 셀을 열어도 재학습하지 않습니다.

In [ ]:
run_scale = False
if run_scale:
    scale_run = train_ablation(baseline, "scale")
    print(scale_run / "weights/best.pt")

## 3. scale 결과 확인

아래 두 셀을 실행합니다. 학습 전에는 베이스라인 표를 먼저 봅니다. 학습 후에는 기준 모델과 실험 모델을 같은 39장에 적용합니다. 정탐은 신뢰도 ≥ 0.10이고 정답과 IoU ≥ 0.50인 경우입니다. 영상별 중앙 IoU는 정탐한 프레임만 사용하므로 바로 옆의 미탐 수도 함께 읽습니다. mAP50–95는 학습 중 기록된 검증 곡선의 최고값입니다.

In [ ]:
def compare_with_baseline(trial):
    runs = [baseline]
    if (trial / "weights/best.pt").is_file():
        runs.append(trial)
    else:
        print(f"{trial.name}의 가중치를 기다리는 중입니다.")
    tables = []
    for run in runs:
        history = pd.read_csv(run / "results.csv")
        metric = history["metrics/mAP50-95(B)"]
        print(f"{run.name} · 기록 {len(history)}에폭 · 최고 검증 mAP50–95 {metric.max():.4f}")
        frames = evaluate_plates(YOLO(str(run / "weights/best.pt")), dataset)
        frames["run"] = run.name
        tables.append(frames)
    result = pd.concat(tables, ignore_index=True)
    display(result.groupby(["video", "run"]).agg(frames=("frame", "size"),
                missed=("missed", "sum"), median_hit_iou=("hit_iou", "median")).round(3))
    if trial in runs:
        plot_training_report(trial)
        plt.show()
    return result

In [ ]:
scale_results = compare_with_baseline(scale_run)
display(scale_results[scale_results.video.str.startswith(("a03", "b02"))]
        .pivot(index=["video", "frame", "box_width_640"], columns="run", values="hit_iou").round(3))

## 4. hsv_v 학습

크기 실험을 검토한 뒤 이 셀만 켭니다. 이번 모델도 원래 사전학습 가중치에서 시작하며 크기 증강은 베이스라인의 0.5를 사용합니다. HSV 밝기만 0.4에서 0.0으로 바꿉니다.

In [ ]:
run_hsv_v = False
if run_hsv_v:
    brightness_run = train_ablation(baseline, "hsv_v")
    print(brightness_run / "weights/best.pt")

## 5. hsv_v 결과를 읽고 자세 추정으로 이동

같은 검증 프레임을 비교합니다. 아래 표는 원판 박스와 주변 띠의 V 중앙값 차이가 작은 프레임부터 표시합니다. 장면 전체가 어두운 순서로 보려면 정렬 기준을 `scene_v`로 바꾸세요. 증강의 이득과 손실을 영상·프레임 번호로 기록한 뒤, 가까운 쪽 어깨·골반·무릎·발목의 자세 추정 베이스라인으로 넘어갑니다.

In [ ]:
if (brightness_run / "weights/best.pt").is_file():
    brightness_results = compare_with_baseline(brightness_run)
    by_frame = brightness_results.pivot(index=["video", "frame", "scene_v", "box_background_delta_v"],
                                        columns="run", values="hit")
    display(by_frame.reset_index().sort_values("box_background_delta_v").head(12))
else:
    print("밝기 실험을 마치면 이 셀에서 비교합니다.")